<a href="https://colab.research.google.com/github/ep24b009-HariccharanM/Coding-Exercises/blob/main/Variational_Quantum_Eigensolver_4x4/VQE4x4_failed_attempt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit[visualization] qiskit-ibm-runtime qiskit-aer qiskit_qasm3_import

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter, ParameterVector
import qiskit.qasm3
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import *
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.primitives import StatevectorSampler, PrimitiveJob
import matplotlib.pyplot as plt
import math

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.2/108.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
import numpy as np

# Given matrix
Herm = np.array([[2,0,0,0],[0,5,0,0],[0,0,0,1],[0,0,1,0]])

# Decomposing

I = np.eye(2)
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])

a = np.zeros(16 , dtype = complex)

for i in range(0,4):
  for j in range(0,4):
    if i == 0:
        A = I
    elif i == 1:
        A = X
    elif i == 2:
        A = Y
    else:
        A = Z

    if j == 0:
        B = I
    elif j == 1:
        B = X
    elif j == 2:
        B = Y
    else:
        B = Z

    a[i*4+j]= 0.25* np.trace(Herm @ ( np.kron(A,B)))

print(a)

#Initializing the wavefunction with two parameters
ry1 = 0.0 #Rotation about Y
rz1 = 0.0 #Rotation about z
ry2 = 0.0
rz2 = 0.0

[ 1.75+0.j  0.5 +0.j  0.  +0.j -0.75+0.j  0.  +0.j  0.  +0.j  0.  +0.j
  0.  +0.j  0.  +0.j  0.  +0.j  0.  +0.j  0.  +0.j  1.75+0.j -0.5 +0.j
  0.  +0.j -0.75+0.j]


In [3]:
def anz(ry,rz):
  qc = QuantumCircuit(3,3)
  qc.ry(float(np.real(ry)),0)
  qc.rz(float(np.real(rz)),0)

  qc.ry(float(np.real(ry)),1)
  qc.rz(float(np.real(rz)),1)

  qc.ry(float(np.real(ry)),2)
  qc.rz(float(np.real(rz)),2)

  qc.h(0)
  qc.measure(0,0)

  qc.sdg(1)
  qc.h(1)
  qc.measure(1,1)

  qc.measure(2,2)

  sampler = StatevectorSampler()
  pub = (qc)
  job_sampler = sampler.run([pub], shots=10000)
  result_sampler = job_sampler.result()
  counts_sampler = result_sampler[0].data.c.get_counts()

  total_shots = sum(counts_sampler.values())


  num_zeros_q0 = 0
  for outcome, count in counts_sampler.items():
      if outcome[2] == '0':
          num_zeros_q0 += count


  num_zeros_q1 = 0
  for outcome, count in counts_sampler.items():
      if outcome[1] == '0':
          num_zeros_q1 += count


  num_zeros_q2 = 0
  for outcome, count in counts_sampler.items():
      if outcome[0] == '0':
          num_zeros_q2 += count

  return num_zeros_q0 / total_shots , num_zeros_q1 / total_shots , num_zeros_q2 / total_shots

In [4]:
def f1(ry,rz):
  c_x , c_y , c_z = anz(ry,rz)
  y1 = (a[0]+a[1]+a[2]+a[3]) + (a[4]+a[5]+a[6]+a[7])*(2*c_x-1) + (a[8]+a[9]+a[10]+a[11])*(2*c_y-1) + (a[12]+a[13]+a[14]+a[15])*(2*c_z-1)
  return np.real(y1)

def f2(ry,rz):
  c_x , c_y , c_z = anz(ry,rz)
  y2 = (a[0]+a[4]+a[8]+a[12]) + (a[1]+a[5]+a[9]+a[13])*(2*c_x-1) + (a[2]+a[6]+a[10]+a[14])*(2*c_y-1) + (a[3]+a[7]+a[11]+a[15])*(2*c_z-1)
  return np.real(y2)

def grad_f1(ry, rz): # Assuming that the quantum states are independent and there is no corelation or entanglement.
    df_drx = (f1(ry+0.1,rz) - f1(ry-0.1,rz))/0.2
    df_dry = (f1(ry,rz+0.1) - f1(ry,rz-0.1))/0.2
    return df_drx, df_dry

def grad_f2(ry, rz):
    df_drx = (f2(ry+0.1,rz) - f2(ry-0.1,rz))/0.2
    df_dry = (f2(ry,rz+0.1) - f2(ry,rz-0.1))/0.2
    return df_drx, df_dry

lr = 0.1
steps = 200

for i in range(steps):
    dry1, drz1 = grad_f1(ry1, rz1)
    ry1 -= lr * np.real(dry1)
    rz1 -= lr * np.real(drz1)
    c1 = f1(ry1, rz1)

    dry2, drz2 = grad_f2(ry2, rz2)
    ry2 -= lr * np.real(dry2)
    rz2 -= lr * np.real(drz2)
    c2 = f2(ry2, rz2)
    print(f"Step {i+1}: ry1={ry1:.4f}, rz1={rz1:.4f}, c1={c1:.4f},ry2={ry2:.4f}, rz2={rz2:.4f}, c2={c2:.4f}")

print(f"Minimum at ry1={ry1:.4f}, rz1={rz1:.4f}, c1={c1:.4f} and ")
print(f"Minimum at ry2={ry2:.4f}, rz2={rz2:.4f}, c2={c2:.4f}")

Step 1: ry1=-0.0005, rz1=0.0000, c1=2.0000,ry2=-0.0002, rz2=0.0000, c2=2.0000
Step 2: ry1=-0.0008, rz1=0.0000, c1=2.0000,ry2=-0.0009, rz2=0.0000, c2=2.0000
Step 3: ry1=-0.0008, rz1=0.0000, c1=2.0000,ry2=-0.0003, rz2=0.0000, c2=2.0000
Step 4: ry1=-0.0009, rz1=0.0000, c1=2.0000,ry2=0.0009, rz2=0.0000, c2=2.0000
Step 5: ry1=-0.0010, rz1=0.0000, c1=2.0000,ry2=-0.0008, rz2=0.0000, c2=2.0000
Step 6: ry1=-0.0012, rz1=0.0000, c1=2.0000,ry2=0.0009, rz2=0.0000, c2=2.0000
Step 7: ry1=-0.0008, rz1=0.0000, c1=2.0000,ry2=0.0021, rz2=0.0000, c2=2.0000
Step 8: ry1=-0.0006, rz1=0.0000, c1=2.0000,ry2=0.0018, rz2=0.0000, c2=2.0000
Step 9: ry1=-0.0005, rz1=0.0000, c1=2.0000,ry2=0.0004, rz2=0.0000, c2=2.0000
Step 10: ry1=-0.0002, rz1=0.0000, c1=2.0000,ry2=0.0004, rz2=0.0000, c2=2.0000
Step 11: ry1=-0.0004, rz1=0.0000, c1=2.0000,ry2=-0.0002, rz2=0.0000, c2=2.0000
Step 12: ry1=-0.0012, rz1=0.0000, c1=2.0000,ry2=-0.0008, rz2=0.0000, c2=2.0000
Step 13: ry1=-0.0011, rz1=0.0000, c1=2.0000,ry2=-0.0008, rz2=0.0000

In [5]:
fin_ev1 = np.array([np.cos(ry1/2), np.sin(ry1/2)*np.exp(1j*rz1)])
fin_ev2 = np.array([np.cos(ry2/2), np.sin(ry2/2)*np.exp(1j*rz2)])
fin_ev = np.kron(fin_ev1,fin_ev2)
print("The eigen vector that has the least eigen value is\n",fin_ev)

The eigen vector that has the least eigen value is
 [9.90166917e-01+0.00000000e+00j 5.94100195e-04+1.78230064e-07j
 1.39889617e-01+1.04917233e-04j 8.39337577e-05+8.81304780e-08j]


# **Reason For Failure**

I have taken a quantum circuit where both the qubits are independent. I have not considered the case where they can be entangled. Due to this a large portion of space of all Double qubits has been excluded. Hence it is impossible for this code to converge at the right state.